# Solutions — Notebook 05: Comparison and Capstone

**BINFX410 — Chapter 10**

> **Prerequisite:** Run notebooks 01–04 in order. This solution notebook expects:
> - `../lakehouse/` Delta tables (from notebook 04)
> - `../figures/` directory will be created automatically if it does not exist

In [ ]:
import os
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
import duckdb
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['figure.dpi'] = 100

con = duckdb.connect()
con.execute('INSTALL delta; LOAD delta;')

os.makedirs('../figures', exist_ok=True)
print('Setup complete.')

## Capstone Extension 5.1 — Highest-Confidence Somatic Calls

Dr. Kim asks: *"Which variants have the highest quality scores AND highest allele frequency? These are our highest-confidence somatic calls."*

Write the query, generate a scatter plot (`af_tumor` vs `quality_score`, colored by `consequence`), and write a 2–3 sentence interpretation of what you see.

In [ ]:
# Highest-confidence somatic calls: high quality_score AND high af_tumor (PASS only)
high_conf = con.execute("""
    SELECT
        v.variant_id,
        v.af_tumor,
        v.quality_score,
        v.consequence,
        v.variant_type,
        g.gene_name,
        g.is_cancer_gene
    FROM delta_scan('../lakehouse/variants')      v
    JOIN delta_scan('../lakehouse/genes')         g  ON v.gene_id  = g.gene_id
    JOIN delta_scan('../lakehouse/variant_calls') vc ON v.call_id  = vc.call_id
    WHERE vc.filter_status = 'PASS'
""").df()

# Define threshold: top quartile of BOTH metrics
af_thresh = high_conf['af_tumor'].quantile(0.75)
qs_thresh = high_conf['quality_score'].quantile(0.75)
top_calls  = high_conf[
    (high_conf['af_tumor'] >= af_thresh) & (high_conf['quality_score'] >= qs_thresh)
]

print(f'Total PASS variants          : {len(high_conf):,}')
print(f'High-confidence calls (top Q): {len(top_calls):,}')
print(f'  af_tumor >= {af_thresh:.2f}, quality_score >= {qs_thresh:.1f}')

# Scatter plot
consequence_colors = {
    'missense_variant':    '#e74c3c',
    'stop_gained':         '#8e44ad',
    'frameshift_variant':  '#2c3e50',
    'synonymous_variant':  '#95a5a6',
    'splice_site_variant': '#e67e22',
    'intron_variant':      '#3498db',
}

fig, ax = plt.subplots(figsize=(9, 6))
for csq, grp in high_conf.groupby('consequence'):
    ax.scatter(grp['af_tumor'], grp['quality_score'],
               alpha=0.4, s=18, label=csq,
               color=consequence_colors.get(csq, '#7f8c8d'))

ax.axvline(af_thresh,  color='red',    linestyle='--', linewidth=1, label=f'af_tumor Q75 ({af_thresh:.2f})')
ax.axhline(qs_thresh,  color='orange', linestyle='--', linewidth=1, label=f'quality Q75 ({qs_thresh:.1f})')
ax.set_xlabel('Allele Frequency in Tumor (af_tumor)')
ax.set_ylabel('Quality Score (Phred-scaled)')
ax.set_title('Variant Confidence Landscape\nPASS calls — colored by consequence')
ax.legend(fontsize=8, loc='upper left')
plt.tight_layout()
plt.savefig('../figures/capstone_q6_confidence_scatter.png', dpi=150)
plt.show()
print(f'Saved to ../figures/capstone_q6_confidence_scatter.png')

print("""
Interpretation:
Variants in the upper-right quadrant (high af_tumor AND high quality_score) are our
highest-confidence somatic calls. These represent variants with strong sequencing
evidence (high quality) that are clearly clonal or sub-clonal (high allele frequency).
stop_gained and frameshift variants in this quadrant are strong candidates for
functional driver mutations — particularly if they fall in known cancer genes.
Synonymous variants with high af_tumor but lower quality may reflect sequencing
artefacts and should be treated cautiously.
""")

## Capstone Extension 5.2 — Scale Test

In Notebook 01, re-generate the dataset with **10x the data** (5,000 samples, 20,000 variant_calls, 50,000 variants). Re-run the benchmarks in this notebook. How do the relative performance differences between CSV, Parquet, DuckDB warehouse, and Delta Lake change?

---

To run this exercise:

1. Open `01_introduction_and_setup.ipynb` and change these constants at the top of the data generation section:
   ```python
   N_SAMPLES  = 5000   # was 500
   N_GENES    = 100    # keep the same
   N_CALLS    = 20000  # was 2000
   N_VARIANTS = 50000  # was 5000
   ```
2. Re-run notebook 01, then re-run notebooks 02–04 to rebuild all layers.
3. Re-run the benchmark cells in notebook 05.

**Expected observations:**

| Format | 5K rows | 50K rows | Why |
|--------|---------|---------|-----|
| CSV (pandas) | fast | Slower — full scan, string parsing | No compression, row-oriented |
| Parquet (DuckDB) | fast | Still fast — column pruning helps | Columnar, compressed, predicate pushdown |
| DuckDB warehouse | fast | Fastest — indexes, columnar, vectorized | Purpose-built for OLAP, statistics in catalog |
| Delta Lake | fast | Fast — same Parquet underneath | Minor overhead from log read at startup |

At 50K rows the differences become visible. At 500K–5M rows, the CSV/pandas approach risks running out of RAM on a laptop while Parquet and DuckDB continue to scale. The key insight is that Parquet and DuckDB warehouse perform similarly at small scales but the warehouse gains more advantage as data grows due to statistics-based query optimization.

In [ ]:
# To run the scale test: modify constants in notebook 01, then re-run the full pipeline.
# N_SAMPLES  = 5000
# N_CALLS    = 20000
# N_VARIANTS = 50000

# Then re-run benchmarks in notebook 05 and compare results.
print('Run notebook 01 with larger constants, re-run notebooks 02-04, then re-run notebook 05 benchmarks.')

## Capstone Extension 5.3 — Lakehouse vs. Warehouse for Clinical Genomics

A hospital wants to use variant data for two purposes:

1. **Clinical reporting**: structured SQL queries for monthly somatic mutation burden reports per patient, delivered to oncologists. Strict data governance required. Schema must be stable.

2. **Research**: ML models for pathogenicity scoring that require raw variant features, schema flexibility, and the ability to add new annotation columns as databases are updated.

Design the architecture that best serves both use cases.

---

### Architecture Recommendation: Lakehouse

**Why not a pure warehouse?**
A warehouse (DuckDB/Snowflake) handles SQL reporting well but is poorly suited for the research workload: ML training requires loading data into Python DataFrames with arbitrary features, novel annotation columns need to be added frequently as new databases (ClinVar, gnomAD) are released, and raw file access (BAM/VCF inspection) is impossible from within a warehouse schema.

**Why not a pure data lake?**
Clinical reporting requires ACID guarantees — a partially-written batch of variant calls must never be visible to a report. Schema-on-read means data quality problems reach clinicians instead of being caught at ingest. There is no audit trail of what data was present when a clinical report was generated.

**Lakehouse design:**

```
Bronze  →  Silver  →  Gold (Clinical)       Gold (Research)
──────     ──────     ──────────────         ───────────────
Raw VCF    Validated  Strict schema          Flexible schema
BAM files  variants   ACID transactions      Schema evolution
           Filtered   Regulatory reporting   ML feature tables
                      Time travel = audit    Pathogenicity scores
```

**Key properties needed:**
- **ACID + time travel** (Delta Lake): every clinical report can be reproduced exactly by querying the table at the version that existed when the report was generated.
- **Schema enforcement** on the Silver → Gold (Clinical) path: prevents pipeline bugs from reaching clinical output.
- **Schema evolution** on the Gold (Research) path: new annotation sources (gnomAD v4, AlphaMissense) add columns via `schema_mode='merge'` without breaking existing pipelines.
- **Same underlying files**: clinical SQL (DuckDB `delta_scan()`) and ML training (`.to_pandas()`) both read the same Parquet files — no ETL duplication.

**Data flow:**

```
Sequencer → VCF/BAM files
         → Bronze Delta table (raw, append-only, schema enforcement OFF)
         → Silver Delta table (validated, PASS filter, schema enforcement ON)
         → Gold (Clinical) Delta table  ──►  DuckDB SQL → Monthly reports
         → Gold (Research) Delta table  ──►  pandas / PyTorch → ML models
```

**Implementation (local):** Delta Lake + DuckDB is sufficient for a pilot. Production scale would use Delta Lake on cloud object storage (S3/ADLS) with a query engine like Trino or Spark SQL for the clinical layer and pandas/PyTorch for the research layer.